# Recitation 4: Regression with scikit-learn
### Data Science for Entrepreneurship-Economy | Spring 2026

**Today's goal**: Build and compare regression models that predict a continuous number — from a single feature to polynomial curves to regularized models.

By the end of this session you'll be able to:
- Build simple and multiple linear regression models
- Interpret coefficients as business-meaningful "price tags"
- Fit polynomial curves and diagnose overfitting
- Apply Ridge and Lasso regularization
- Use cross-validation for reliable model evaluation

**Time**: ~90 minutes | **Exercises**: 2 | **Format**: Guided walkthrough — follow along, then practice

**How this session works**: Same as Recitation 3 — the instructor drives the pipeline, you run each cell, and at key decision points you'll get exercises.

### 📖 Quick Recap

- In Recitation 3 you built classifiers that predict **labels** (churn / no-show). Today you predict **numbers**.
- The scikit-learn pattern is identical: `create → .fit() → .predict()`
- New metrics: MAE, RMSE, R² replace accuracy/precision/recall
- New tools: polynomial features, Ridge, Lasso, cross-validation
- The thinking is the same: prepare data → train → evaluate → compare → decide

Last week you predicted *yes/no*. This week you predict *how much*.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# scikit-learn imports for regression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")

In [ ]:
# Load the AutoBid used-car dataset
df = pd.read_csv("autobid_used_cars.csv")

> **AutoBid** is an Israeli online used-car marketplace. The platform connects private sellers with buyers, but sellers often struggle to price their cars correctly — overpricing leads to weeks without offers, underpricing leaves money on the table. AutoBid's product lead, Dana, wants to build a **price prediction engine** that suggests a fair listing price based on car attributes. This would help sellers list faster and help buyers trust that prices are fair.

Dana needs a pricing engine that sellers trust. Let's build one.

In [ ]:
print(f"Dataset: {df.shape[0]} listings × {df.shape[1]} columns\n")
print(df.describe().round(1))

In [ ]:
df["listing_price_thousands"].hist(bins=20, color="#214073", edgecolor="white")
plt.xlabel("Listing Price (₪K)")
plt.ylabel("Number of Cars")
plt.title("Distribution of Used Car Prices — AutoBid")
plt.show()

print(f"Mean price: ₪{df['listing_price_thousands'].mean():.0f}K")
print(f"Median price: ₪{df['listing_price_thousands'].median():.0f}K")

Right-skewed — most cars cluster in the middle range, with a tail of luxury and electric vehicles stretching higher. Mean > median confirms the skew.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df["car_age_years"], df["listing_price_thousands"], alpha=0.4, color="#214073")
axes[0].set_xlabel("Car Age (years)")
axes[0].set_ylabel("Price (₪K)")
axes[0].set_title("Age vs. Price — Linear?")

axes[1].scatter(df["km_driven_thousands"], df["listing_price_thousands"], alpha=0.4, color="#008C8C")
axes[1].set_xlabel("KM Driven (thousands)")
axes[1].set_ylabel("Price (₪K)")
axes[1].set_title("KM vs. Price")

axes[2].scatter(df["engine_size_cc"], df["listing_price_thousands"], alpha=0.4, color="#E6592D")
axes[2].set_xlabel("Engine Size (cc)")
axes[2].set_ylabel("Price (₪K)")
axes[2].set_title("Engine Size vs. Price — Curved?")

plt.tight_layout()
plt.show()

Age vs. price is clearly linear (downward). Engine size vs. price shows a curve — rises steeply then flattens. That nonlinearity will matter later.

In [ ]:
print("Correlation between car_age and km_driven:")
print(f"  r = {df['car_age_years'].corr(df['km_driven_thousands']):.2f}")

Older cars have been driven more — these two features are tangled (multicollinearity). We'll see how this affects our model in later sections.

In [ ]:
X = df.drop(columns=["listing_price_thousands", "listing_id"])
y = df["listing_price_thousands"]

# One-hot encode categorical columns (BMW is baseline — dropped first alphabetically)
X = pd.get_dummies(X, columns=["fuel_type", "car_brand", "city"], drop_first=True)
print(f"Features after encoding: {X.shape[1]} columns")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]}")

---
## 📏 Section 2: Simple Linear Regression
*One feature, one line. How far can car age alone take us?*

---

In [ ]:
# Use ONLY car_age_years
X_train_age = X_train[["car_age_years"]]
X_test_age = X_test[["car_age_years"]]

simple_model = LinearRegression()
simple_model.fit(X_train_age, y_train)

print(f"Slope (β₁):     {simple_model.coef_[0]:.2f}")
print(f"Intercept (β₀): {simple_model.intercept_:.2f}")

### 📖 Interpreting the Coefficients

Read the slope as depreciation per year: each additional year of age drops the predicted price. The intercept is the predicted price of a brand-new car (age = 0).

Notice the slope absorbs the effect of km_driven too — older cars have more km, and since km isn't in the model, age takes credit for both effects. We'll see this change in multiple regression.

In [ ]:
y_pred_simple = simple_model.predict(X_test_age)

r2_simple = r2_score(y_test, y_pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test, y_pred_simple))
mae_simple = mean_absolute_error(y_test, y_pred_simple)

print(f"Simple Regression (car_age only):")
print(f"  R²:   {r2_simple:.3f}")
print(f"  RMSE: ₪{rmse_simple:.1f}K")
print(f"  MAE:  ₪{mae_simple:.1f}K")

In [ ]:
plt.scatter(X_test_age, y_test, alpha=0.5, label="Actual", color="#214073")
age_sorted = np.sort(X_test_age.values, axis=0)  # sorted for a clean line
plt.plot(age_sorted, simple_model.predict(age_sorted),
         color="#E6592D", linewidth=2, label="Prediction line")
plt.xlabel("Car Age (years)")
plt.ylabel("Price (₪K)")
plt.title(f"Simple Regression | R² = {r2_simple:.2f}")
plt.legend()
plt.show()

### 🎯 Exercise 1: Predict and Interpret

- A seller has a **7-year-old car**. Using the simple model's equation (slope and intercept from above), calculate the predicted price **by hand** (mental math).
- Then verify with `simple_model.predict([[7]])`.
- The actual listing price is ₪82K. What's the residual? Did the model over- or underestimate?
- Is R² good enough for Dana's pricing engine? Why or why not?

In [ ]:
##### Your code goes here #####

# 1. Manual calculation: Price = intercept + slope × 7

# 2. Verify with model

# 3. Residual = actual - predicted. Over or under?

# 4. Is this R² good enough? (comment)

###### End of Your Code ######

### ✅ Solution

<details>
<summary>Click to reveal</summary>

```python
# Manual: intercept + slope * 7
manual_pred = simple_model.intercept_ + simple_model.coef_[0] * 7
print(f"Manual prediction: ₪{manual_pred:.1f}K")

# Verify
model_pred = simple_model.predict([[7]])[0]
print(f"Model prediction:  ₪{model_pred:.1f}K")

# Residual
actual = 82
residual = actual - model_pred
print(f"Residual: {residual:+.1f}K ({'underestimated' if residual > 0 else 'overestimated'})")

# R² ~ 0.55 means 45% of price variation is unexplained.
# RMSE ~ ₪34K means typical errors are huge — not reliable enough for a pricing tool.
print(f"\nR² = {r2_simple:.2f} — too rough for a pricing engine. Need more features.")
```

</details>

### 💡 Insight

Car age alone explains about half the price variation. A typical error of ₪30K+ is too big for a pricing tool — a seller following this would misprice significantly. We need more features.

---
## 📊 Section 3: Multiple Regression
*Add all features. Every variable gets its own "price tag."*

---

In [ ]:
full_model = LinearRegression()
full_model.fit(X_train, y_train)
y_pred_full = full_model.predict(X_test)

r2_full = r2_score(y_test, y_pred_full)
rmse_full = np.sqrt(mean_squared_error(y_test, y_pred_full))

print(f"Multiple Regression (all features):")
print(f"  R²:   {r2_full:.3f}")
print(f"  RMSE: ₪{rmse_full:.1f}K")
print(f"\nImprovement from simple: R² {r2_simple:.2f} → {r2_full:.2f}")

In [ ]:
coefs = pd.Series(full_model.coef_, index=X_train.columns)
coefs["Intercept"] = full_model.intercept_

# Show top coefficients by absolute value
print("Top coefficients (sorted by |value|):\n")
top = coefs.drop("Intercept").abs().sort_values(ascending=False).head(10)
for feat in top.index:
    print(f"  {feat:30s}: {coefs[feat]:+.2f} ₪K")
print(f"\n  {'Intercept':30s}: {coefs['Intercept']:.2f} ₪K")

### 📖 Interpreting the Coefficients

Read each coefficient as: "all else equal, this feature changes the predicted price by this much."

Since BMW is the **baseline brand** (dropped by one-hot encoding), the brand coefficients show how much cheaper each brand is compared to BMW. For example, `car_brand_Hyundai: −₪44K` means a Hyundai is predicted to be ₪44K cheaper than an equivalent BMW. This confirms BMW's premium status.

Similarly, `fuel_type_Electric: +₪35K` means electric cars are ₪35K more expensive than the baseline fuel type (Diesel).

Notice that `car_age_years` coefficient shrank compared to simple regression. That's because `km_driven_thousands` now absorbs some of what age was explaining alone — they share information.

In [ ]:
# Compare age coefficient: simple vs. multiple
print(f"Simple model — car_age slope: {simple_model.coef_[0]:+.2f}")
print(f"Multiple model — car_age:     {coefs['car_age_years']:+.2f}")
print(f"Multiple model — km_driven:   {coefs['km_driven_thousands']:+.4f}")

### 💬 Instructor Discussion: The Multicollinearity Question

> **Discuss with the class:**
>
> - Why did the `car_age_years` coefficient shrink when we added all features?
> - If you removed `km_driven_thousands`, what would happen to the age coefficient?
> - Should Dana include BOTH age and km, or just one?

<details>
<summary>Key points</summary>

- `car_age` shrank because `km_driven` now captures some of the same effect. These features are correlated (r ≈ 0.75) — they share information, and the shared variance gets split between them.
- If we removed `km_driven`, `car_age` would grow back toward its simple regression value, absorbing the km effect again.
- Keep both — the model's R² is higher with both features. Regularization (Section 5) handles the multicollinearity more elegantly.

</details>

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred_full, alpha=0.5, color="#214073")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         "r--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual Price (₪K)")
plt.ylabel("Predicted Price (₪K)")
plt.title(f"Multiple Regression — Actual vs. Predicted | R² = {r2_full:.2f}")
plt.legend()
plt.show()

### 💡 Insight

R² jumped substantially — adding brand, fuel type, km, and other features captured much of the remaining variation. Points cluster closer to the red line. For a rough pricing tool, this is getting usable.

---
## 🔄 Section 4: Polynomial Regression
*When straight lines aren't enough — fitting curves to capture nonlinear patterns.*

---

### 📖 Why Polynomial?

Remember the engine_size scatter plot from Section 1? Price rises steeply for small engines, then flattens for large ones. A straight line can't capture that curve.

Polynomial regression adds x² (and higher powers) as features so the model can fit curves. We'll apply it **to engine_size specifically** — because that's the feature with the visible nonlinearity. Adding polynomial terms to every feature would create dozens of extra columns that just add noise.

In [ ]:
# PolynomialFeatures on engine_size only
poly = PolynomialFeatures(degree=2, include_bias=False)
engine_train = poly.fit_transform(X_train[["engine_size_cc"]])
engine_test = poly.transform(X_test[["engine_size_cc"]])
print(f"Poly features: {poly.get_feature_names_out()}")

In [ ]:
# Replace engine_size_cc with engine + engine²
X_train_poly = X_train.drop(columns=["engine_size_cc"])
X_test_poly = X_test.drop(columns=["engine_size_cc"])
X_train_poly["engine_cc"] = engine_train[:, 0]
X_train_poly["engine_cc²"] = engine_train[:, 1]
X_test_poly["engine_cc"] = engine_test[:, 0]
X_test_poly["engine_cc²"] = engine_test[:, 1]

print(f"Features: {X_train_poly.shape[1]} (added engine_cc²)")

In [ ]:
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)

r2_poly = r2_score(y_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))

print(f"Polynomial (engine² added):")
print(f"  R²:   {r2_poly:.3f}")
print(f"  RMSE: ₪{rmse_poly:.1f}K")
print(f"\nImprovement from linear: R² {r2_full:.2f} → {r2_poly:.2f}")

Adding engine_size² captured the curved relationship — a modest but real improvement. The model now knows that a 2000cc engine isn't just "twice as good" as 1000cc; the benefit plateaus for large engines.

### ⚠️ Common Mistake: More Polynomial = Better?

What if we go further? Let's add polynomial features to ALL numeric columns and increase the degree.

In [ ]:
# Polynomial on ALL numeric features — degrees 1, 2, 3
num_cols = ["car_age_years", "km_driven_thousands", "engine_size_cc",
            "num_previous_owners", "has_sunroof"]
dum_cols = [c for c in X_train.columns if c not in num_cols]

print(f"{'Degree':<8} {'Features':<10} {'R² (train)':<12} {'R² (test)':<12}")
print("-" * 42)
for degree in [1, 2, 3]:
    pf = PolynomialFeatures(degree=degree, include_bias=False)
    Xtr_p = np.hstack([pf.fit_transform(X_train[num_cols]),
                        X_train[dum_cols].values])
    Xte_p = np.hstack([pf.transform(X_test[num_cols]),
                        X_test[dum_cols].values])
    m = LinearRegression().fit(Xtr_p, y_train)
    r2_tr = m.score(Xtr_p, y_train)
    r2_te = r2_score(y_test, m.predict(Xte_p))
    print(f"{degree:<8} {Xtr_p.shape[1]:<10} {r2_tr:<12.3f} {r2_te:<12.3f}")

Degree 3 on all numeric features fits training data better — but test R² drops. It memorized the training data, including the noise. Classic **overfitting**.

The sweet spot was our targeted approach: add engine² for the one feature that actually has a curve. Blindly adding polynomial terms to everything creates noise, not signal.

### 💬 Instructor Discussion: Diagnose the Overfitting

> **Discuss with the class:**
>
> - At which degree does overfitting start? Where does the train-test gap widen dramatically?
> - If Dana deployed the degree-3 all-features model, what would happen when a new car is listed?
> - Why is the targeted engine² approach better? Think in business terms.

<details>
<summary>Key points</summary>

- Overfitting starts at degree 3 — training R² keeps climbing but test R² drops. The gap = memorizing, not learning.
- If Dana deployed degree-3-all, the model would give wildly wrong prices for unseen cars — sellers would lose trust instantly.
- The targeted engine² approach works because we **know** engine_size has a curve (scatter plot). We're adding information, not noise. In business: "Big engines have diminishing returns on price — buyers don't pay proportionally more for 3.5L vs 2.0L."

</details>

### 💡 Insight

In business, a model that's "almost perfect" on historical data but crashes on new data is worse than a simpler model that's consistently decent. Dana wants reliability, not perfection.

---
## 🛡️ Section 5: Regularization — Ridge & Lasso
*Keeping coefficients under control. Lasso even drops useless features automatically.*

---

### 📖 Why Regularization?

The polynomial overfitting happened because coefficients grew huge to fit every training point. Regularization adds a penalty: the model must fit the data AND keep coefficients small.

- **Ridge (L2)**: shrinks all coefficients toward zero — none disappear completely
- **Lasso (L1)**: can shrink some coefficients all the way to zero — automatic feature selection

**Important**: Ridge and Lasso penalize coefficients by size. If features have different scales (engine_size_cc is 1000–3500 while has_sunroof is 0–1), the penalty hits them unequally. We must **standardize** first so the penalty is fair.

> **A note on scaling**: In practice, standardization (e.g. `StandardScaler`) is recommended even for plain linear regression — it helps with numerical stability, convergence, and makes coefficients comparable across features. We skipped it in Sections 2–4 deliberately so you could read coefficients in their original units (₪K per year, ₪K per 1000 km, etc.) and build intuition. From here on we scale properly.

In [ ]:
# Scale features so regularization penalty is fair across all features
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train),
                          columns=X_train.columns, index=X_train.index)
X_test_sc = pd.DataFrame(scaler.transform(X_test),
                         columns=X_test.columns, index=X_test.index)

In [ ]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_sc, y_train)
y_pred_ridge = ridge_model.predict(X_test_sc)

r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

print(f"Ridge (α=1.0):")
print(f"  R²:   {r2_ridge:.3f}")
print(f"  RMSE: ₪{rmse_ridge:.1f}K")

In [ ]:
lasso_model = Lasso(alpha=0.5)
lasso_model.fit(X_train_sc, y_train)
y_pred_lasso = lasso_model.predict(X_test_sc)

r2_lasso = r2_score(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

print(f"Lasso (α=0.5):")
print(f"  R²:   {r2_lasso:.3f}")
print(f"  RMSE: ₪{rmse_lasso:.1f}K")

In [ ]:
lasso_coefs = pd.Series(lasso_model.coef_, index=X_train_sc.columns)

# Features Lasso kept (non-zero)
kept = lasso_coefs[lasso_coefs != 0].sort_values(key=abs, ascending=False)
dropped = lasso_coefs[lasso_coefs == 0]

print(f"Lasso KEPT {len(kept)} features, DROPPED {len(dropped)}:\n")
print("Kept (coefficients on standardized features):")
for feat, val in kept.items():
    print(f"  {feat:30s}: {val:+.2f}")
print(f"\nDropped (coefficient = 0):")
for feat in dropped.index:
    print(f"  {feat}")

### 📖 Why Lasso Drops Features

Lasso decided that `has_sunroof` and some other features don't add enough predictive value to justify their coefficient. It zeroed them out — effectively saying "these don't matter once you know the important features." This is automatic feature selection, and it makes the model simpler and more interpretable.

### 🎯 Exercise 2: Ridge vs. Lasso Coefficients

- Compare the `car_age_years` and `km_driven_thousands` coefficients for Ridge and Lasso (both trained on standardized features, so they're directly comparable)
- Which model shrinks coefficients more aggressively?
- Why does regularization help with multicollinearity?

In [ ]:
##### Your code goes here #####

# Print car_age and km_driven coefficients for Ridge and Lasso
# Hint: ridge_model.coef_, lasso_coefs (both on standardized features)

# Which model has more stable (smaller) coefficients?

# Why does regularization help with multicollinearity? (comment)

###### End of Your Code ######

### ✅ Solution

<details>
<summary>Click to reveal</summary>

```python
ridge_coefs = pd.Series(ridge_model.coef_, index=X_train_sc.columns)

print(f"{'Model':<15} {'car_age':>10} {'km_driven':>12}")
print("-" * 37)
print(f"{'Ridge':<15} {ridge_coefs['car_age_years']:>10.2f} {ridge_coefs['km_driven_thousands']:>12.2f}")
print(f"{'Lasso':<15} {lasso_coefs['car_age_years']:>10.2f} {lasso_coefs['km_driven_thousands']:>12.2f}")

# Because we standardized, these coefficients are directly comparable.
# Ridge shrinks both, distributing the shared effect more evenly.
# Without regularization, correlated features "fight for credit" — coefficients
# can bounce wildly. Regularization resolves the ambiguity by penalizing large values.
```

</details>

### 💡 Insight

Regularization isn't just about overfitting — it also stabilizes models with correlated features. For Dana, this means the pricing engine gives consistent results even if the training data shifts slightly.

---
## 🔁 Section 6: Cross-Validation
*Don't trust one exam. Take five.*

---

### 📖 Why Cross-Validation?

Every score we've reported depends on which 20% ended up in the test set. A different random split could give different numbers. Cross-validation trains and tests on multiple splits, averaging the results for a more reliable estimate.

In [ ]:
# 5-fold CV on the full linear model
cv_scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring="r2")

print("5-Fold Cross-Validation (Linear Regression):")
print(f"  R² per fold: {cv_scores.round(3)}")
print(f"  Mean R²:     {cv_scores.mean():.3f}")
print(f"  Std:         {cv_scores.std():.3f}")

In [ ]:
from sklearn.pipeline import make_pipeline

models = {
    "Linear Regression": LinearRegression(),
    "Ridge (α=1.0)": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso (α=0.5)": make_pipeline(StandardScaler(), Lasso(alpha=0.5)),
}

print(f"{'Model':<25} {'Mean R²':<12} {'Std':<10}")
print("-" * 47)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="r2")
    print(f"{name:<25} {scores.mean():<12.3f} {scores.std():<10.3f}")

### 💬 Instructor Discussion: Why Cross-Validation Matters

> **Discuss with the class:**
>
> - Compare `r2_full` (single split) with the CV mean R². Are they close?
> - Which number do you trust more? Why?
> - If you were reporting to Dana, which would you use?

<details>
<summary>Key points</summary>

- The single split might be slightly higher or lower than CV — that's the point. CV averages over randomness, giving a more reliable estimate.
- Report to Dana: "Our model has R² ≈ 0.76 ± 0.03" — the ± shows reliability. A single number without uncertainty is misleading.

</details>

---
## 🏆 Section 7: The Grand Comparison
*Five models. One recommendation for Dana.*

---

In [ ]:
results = []
model_preds = {
    "Simple (age only)": y_pred_simple,
    "Multiple (all features)": y_pred_full,
    "Poly (engine²)": y_pred_poly,
    "Ridge (α=1.0)": y_pred_ridge,
    "Lasso (α=0.5)": y_pred_lasso,
}

for name, preds in model_preds.items():
    results.append({
        "Model": name,
        "R²": r2_score(y_test, preds),
        "RMSE (₪K)": np.sqrt(mean_squared_error(y_test, preds)),
        "MAE (₪K)": mean_absolute_error(y_test, preds),
    })

results_df = pd.DataFrame(results).set_index("Model")
results_df = results_df.round(2)
print(results_df.to_string())

In [ ]:
results_df["R²"].plot(kind="barh", figsize=(8, 4), color="#214073", legend=False)
plt.title("Model Comparison — R² Score")
plt.xlabel("R²")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

### 💬 Instructor Discussion: Which Model for Dana?

> **Part 1 — Model selection:**
> Dana needs **accuracy** (reasonable RMSE) and **interpretability** (explain to sellers WHY the price is what it is).
> Which models perform best on accuracy? Which are also interpretable? What's your recommendation?
>
> **Part 2 — 3-minute CEO pitch:**
> 1. **The model**: Which model and why? State the key metric (R², RMSE).
> 2. **Top factors**: What drives used car prices most? Top 3 with ₪ impact.
> 3. **A surprise**: What counterintuitive pattern did the data reveal?
> 4. **Limitation**: What can't the model predict? What should Dana collect next?

<details>
<summary>Key points</summary>

**Model selection:**
- Multiple LR, Poly (engine²), Ridge, and Lasso all perform well. Simple LR is clearly not enough.
- **Multiple LR and Ridge** are most interpretable — each feature has a clear "price tag" coefficient Dana can show sellers. Lasso is good if Dana wants fewer features.
- **Recommendation: Ridge for production** — stable, accurate, interpretable, handles multicollinearity gracefully. Lasso as secondary analysis to identify which features truly matter.

**CEO pitch:**
1. **Ridge regression** — reliable and interpretable. CV R² ≈ 0.76, meaning it explains ~76% of price variation consistently.
2. **Top factors**: Brand (BMW premium: other brands ₪20-45K cheaper), fuel type (Electric +₪30K), car age (−₪5-10K/year).
3. **Surprise**: `has_sunroof` barely matters — Lasso dropped it entirely. Sellers who think sunroofs add major value are wrong.
4. **Limitations**: Can't see condition, accident history, or photos. Adding a condition rating and photo quality score could improve predictions substantially.

</details>

---

### 🎓 What You Just Did

- Loaded 350 used-car listings and explored price drivers with EDA
- Built simple regression (1 feature) and multiple regression (all features)
- Added a targeted polynomial feature (engine²) and diagnosed overfitting at higher degrees
- Applied Ridge and Lasso regularization — and watched Lasso drop useless features
- Used 5-fold cross-validation for reliable model scoring
- Compared 5 models and recommended one for a real business use case

### The Regression Toolkit

```python
LinearRegression()                    # straight lines
PolynomialFeatures(degree=2)          # add curves (targeted)
Ridge(alpha=1.0)                      # stable coefficients
Lasso(alpha=0.5)                      # feature selection
cross_val_score(model, X, y, cv=5)    # reliable evaluation
```

All follow the same API: `create → .fit() → .predict()`. Same pattern as classification. Muscle memory.

### 📅 What's Next

- **Lecture 5**: Clustering — finding hidden groups in unlabeled data (unsupervised learning)

### 💬 One Last Thing

Last week you classified. This week you predicted prices. You've now mastered both flavors of supervised learning. Next week: what happens when there are NO labels? The algorithm finds structure on its own. That's unsupervised learning — and it's where things get really interesting.